# Embedding-similarity AOI finder

Pick one or more **reference clearcut points**, choose an **area of interest** (all of Florida, or selected counties), and the tool finds every pixel whose AlphaEarth embedding is similar to your references, returning a **vector layer**: *here is what you selected* + *here is everything similar in this AOI*. The embedding **year** is a parameter — change it and re-run.

Similarity = **max cosine similarity to any reference** (a pixel matches if it looks like *any* of your exemplars). Vectorization scale adapts to AOI size. Helpers live in `notebooks/clearcut_ag_common.py`.

## 1. Setup

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
_candidates = [cwd / "notebooks", cwd, *[p / "notebooks" for p in cwd.parents], *cwd.parents]
nb_dir = next(p for p in _candidates if (p / "clearcut_ag_common.py").exists())
sys.path.insert(0, str(nb_dir))

import clearcut_ag_common as cac
import geopandas as gpd
import ipywidgets as widgets

REPO_ROOT = cac.find_repo_root(nb_dir)
ee = cac.init_ee()
florida, florida_gdf = cac.load_florida(REPO_ROOT)
OUT_DIR = REPO_ROOT / "data" / "interim" / "similarity_finder"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. AlphaEarth years:", cac.EMBEDDING_COLLECTION)

Ready. AlphaEarth years: GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL


## 2. Parameters
Set reference points as a coord list here, or use the map in §3 to draw them. Set `COUNTY_NAMES = None` for all of Florida, or a list of county names for a smaller AOI.

In [2]:
# Reference clearcut points as (lon, lat). Default: an Ocala-area clearcut.
REFERENCE_POINTS = [
    (-81.8048667, 29.256675),
]

YEAR = 2023            # AlphaEarth annual embedding year (2017-2024)
THRESHOLD = 0.86       # keep pixels with max cosine similarity >= this
AGG = "max"            # "max" (any reference) or "mean" (all references)
MIN_AREA_HA = 1.0      # drop similar patches smaller than this (removes speckle)
COUNTY_NAMES = ["Marion"]   # None -> all of Florida; else list of FL county names

# All 67 Florida county names, for reference:
FL_COUNTY_NAMES = sorted(
    ee.FeatureCollection("TIGER/2018/Counties")
    .filter(ee.Filter.eq("STATEFP", "12")).aggregate_array("NAME").getInfo()
)
print(len(FL_COUNTY_NAMES), "FL counties. First few:", FL_COUNTY_NAMES[:6])

67 FL counties. First few: ['Alachua', 'Baker', 'Bay', 'Bradford', 'Brevard', 'Broward']


## 3. (Optional) Pick reference points on a map
Run this cell, then use the **draw-marker** tool on the map to click clearcut locations. Run the next cell to pull the markers into `REFERENCE_POINTS`. (Skip this if you set the coord list above.)

In [ ]:
import geemap

pick_map = geemap.Map(center=[28.5, -82.0], zoom=7)
pick_map.add_basemap("SATELLITE")
pick_map

In [ ]:
# Pull drawn markers (if any) into REFERENCE_POINTS.
drawn = []
for feat in (getattr(pick_map, "draw_features", None) or []):
    geom = feat.get("geometry", {})
    if geom.get("type") == "Point":
        lon, lat = geom["coordinates"]
        drawn.append((lon, lat))
if drawn:
    REFERENCE_POINTS = cac.normalize_reference_points(drawn)
    print("Using", len(REFERENCE_POINTS), "drawn reference points.")
else:
    print("No markers drawn; keeping REFERENCE_POINTS from §2:", REFERENCE_POINTS)

## 4. Build the AOI and the similarity image

In [5]:
REFERENCE_POINTS = cac.normalize_reference_points(REFERENCE_POINTS)
aoi = cac.counties_aoi(COUNTY_NAMES)  # None -> whole state
area_km2 = aoi.area(maxError=100).divide(1e6).getInfo()
scale = cac.vector_scale_for_area_km2(area_km2)
print(f"AOI: {COUNTY_NAMES or 'all Florida'}  ~{area_km2:,.0f} km2  -> vectorize scale {scale} m")

ref_vecs = cac.reference_vectors(YEAR, REFERENCE_POINTS, florida)
similarity = cac.similarity_image(YEAR, aoi, ref_vecs, agg=AGG)
print(f"{len(REFERENCE_POINTS)} reference points, year {YEAR}, agg={AGG}, threshold={THRESHOLD}")

AOI: ['Marion']  ~4,312 km2  -> vectorize scale 20 m
1 reference points, year 2023, agg=max, threshold=0.86


## 5. Vectorize the similar regions
Polygonize the `similarity >= THRESHOLD` mask. Large/low-threshold results are capped when pulled locally (a warning tells you to raise the threshold or export).

In [6]:
vectors = cac.vectorize_similarity(similarity, aoi, THRESHOLD, scale, min_area_ha=MIN_AREA_HA)
similar_gdf = cac.fc_to_gdf(vectors)
ref_gdf = cac.reference_points_gdf(REFERENCE_POINTS)
if len(similar_gdf):
    similar_gdf["area_ha"] = similar_gdf.to_crs("EPSG:5070").area / 1e4
    print(f"Found {len(similar_gdf)} similar patches (>= {MIN_AREA_HA} ha), "
          f"total {similar_gdf.area_ha.sum():,.0f} ha")
else:
    print("No similar patches at this threshold - try lowering THRESHOLD or MIN_AREA_HA.")

Found 1889 similar patches (>= 1.0 ha), total 24,401 ha


## 6. Map: references + similar regions

In [ ]:
import geemap

result_map = geemap.Map()
result_map.centerObject(aoi, 9)
result_map.add_basemap("SATELLITE")
result_map.addLayer(similarity, {"min": THRESHOLD, "max": 1.0,
                    "palette": ["black", "purple", "magenta", "yellow"]}, "similarity")
if len(similar_gdf):
    result_map.add_gdf(similar_gdf, layer_name="similar regions",
                       style={"color": "#00ffff", "fillOpacity": 0.25})
result_map.add_gdf(ref_gdf, layer_name="reference points",
                   style={"color": "#ff0000"})
result_map

## 7. Export the vector layers (GeoPackage, two layers)

In [8]:
tag = ("_".join(COUNTY_NAMES) if COUNTY_NAMES else "all_FL").replace(" ", "")
gpkg = OUT_DIR / f"similarity_{tag}_{YEAR}_thr{int(THRESHOLD*100)}.gpkg"
ref_gdf.to_file(gpkg, layer="reference_points", driver="GPKG")
if len(similar_gdf):
    similar_gdf.to_file(gpkg, layer="similar_regions", driver="GPKG")
print("Wrote", gpkg)
print("Layers: reference_points (what you selected) + similar_regions (what we found)")

Wrote /home/chazm/projects/artemis-model/data/interim/similarity_finder/similarity_Marion_2023_thr86.gpkg
Layers: reference_points (what you selected) + similar_regions (what we found)


## Notes
- **Change year:** set `YEAR` and re-run §4-§7; each run is one embedding year.
- **Similarity aggregation:** `AGG="max"` finds land like *any* reference; `"mean"` requires similarity to the whole set.
- **AOI size:** the vectorize scale auto-coarsens for large AOIs (20 m county → ~90 m all-Florida) to keep polygon counts manageable. For a big low-threshold run that trips the feature cap, raise `THRESHOLD`/scale or export the mask with `ee.batch.Export`.
- **Speckle:** `MIN_AREA_HA` drops sub-threshold-area patches server-side (a 20 m pixel is 0.04 ha, so without it a mask vectorizes into tens of thousands of specks). Raise it for cleaner, larger patches; lower it to keep small clearcuts.
- **Reference points can sit anywhere in Florida** (sampled from a state-wide embedding); the AOI only bounds where similar land is searched.
- Output GeoPackage carries both layers so downstream GIS shows selection vs. findings.